In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
oos_results = pd.read_csv(
    "outputs/tables/out_of_sample_results.csv"
)

oos_results.head()

,Dataset,Model,VaR_95,VaR_99,ES_97.5,Violations_95,Violation_Rate_95,Violations_99,Violation_Rate_99,Test_N,nu,Expected_Rate_95,Expected_Rate_99,Violation_Error_95,Violation_Error_99,Abs_Calibration_Error_95,Abs_Calibration_Error_99
0,Short_Normal,Single-Dependence Gaussian,0.016895,0.028486,0.029064,19,0.021591,1,0.001136,880,NaN,0.05,0.01,-0.028409,-0.008864,0.028409,0.008864
1,Short_Normal,Regime-Horizon Student-t,0.016514,0.028827,0.029576,21,0.023864,1,0.001136,880,13.0,0.05,0.01,-0.026136,-0.008864,0.026136,0.008864
2,Short_Stress,Single-Dependence Gaussian,0.018361,0.031075,0.032613,1,0.016949,0,0.000000,59,NaN,0.05,0.01,-0.033051,-0.010000,0.033051,0.010000
3,Short_Stress,Regime-Horizon Student-t,0.018423,0.033896,0.037193,1,0.016949,0,0.000000,59,5.0,0.05,0.01,-0.033051,-0.010000,0.033051,0.010000
4,Medium_Normal,Single-Dependence Gaussian,0.008638,0.013508,0.013279,18,0.020455,2,0.002273,880,NaN,0.05,0.01,-0.029545,-0.007727,0.029545,0.007727


In [3]:
dataset_order = [
    "Short_Normal",
    "Short_Stress",
    "Medium_Normal",
    "Medium_Stress",
    "Long_Normal",
    "Long_Stress"
]

In [4]:
def load_test_returns(dataset_name):

    path = (
        f"outputs/tables/"
        f"test_returns_{dataset_name}.csv"
    )

    df = pd.read_csv(path)

    return df["Portfolio_Return"].values

In [5]:
example_returns = load_test_returns(
    "Short_Normal"
)

print(len(example_returns))
print(example_returns[:5])

880
[ 0.00187267  0.00590917 -0.01036071  0.00992313 -0.00373564]


In [6]:
def violation_sequence(
    actual_returns,
    var_value
):

    actual_returns = np.asarray(
        actual_returns
    )

    violations = (
        actual_returns
        <
        -var_value
    ).astype(int)

    return violations

In [7]:
row = oos_results.iloc[0]

actual = load_test_returns(
    row["Dataset"]
)

violations = violation_sequence(
    actual,
    row["VaR_95"]
)

print(
    violations[:20]
)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [8]:
def kupiec_test(
    violations,
    alpha
):

    violations = np.asarray(
        violations
    )

    n = len(violations)

    x = violations.sum()

    if n == 0:
        return {
            "LR_uc": np.nan,
            "p_value": np.nan
        }

    p_hat = x / n

    # Avoid log(0)
    eps = 1e-12

    p_hat = np.clip(
        p_hat,
        eps,
        1 - eps
    )

    alpha = np.clip(
        alpha,
        eps,
        1 - eps
    )

    log_l_null = (
        (n - x) * np.log(1 - alpha)
        +
        x * np.log(alpha)
    )

    log_l_alt = (
        (n - x) * np.log(1 - p_hat)
        +
        x * np.log(p_hat)
    )

    LR_uc = (
        -2
        *
        (
            log_l_null
            -
            log_l_alt
        )
    )

    p_value = (
        1
        -
        stats.chi2.cdf(
            LR_uc,
            df=1
        )
    )

    return {
        "LR_uc": LR_uc,
        "p_value": p_value
    }

In [9]:
kupiec_example = kupiec_test(
    violations,
    alpha=0.05
)

kupiec_example

{'LR_uc': np.float64(18.829740012127075),
 'p_value': np.float64(1.4292113433289622e-05)}

In [10]:
def transition_counts(
    violations
):

    violations = np.asarray(
        violations
    )

    n00 = 0
    n01 = 0
    n10 = 0
    n11 = 0

    for i in range(
        1,
        len(violations)
    ):

        prev = violations[i - 1]
        curr = violations[i]

        if prev == 0 and curr == 0:
            n00 += 1

        elif prev == 0 and curr == 1:
            n01 += 1

        elif prev == 1 and curr == 0:
            n10 += 1

        elif prev == 1 and curr == 1:
            n11 += 1

    return (
        n00,
        n01,
        n10,
        n11
    )

In [11]:
def christoffersen_independence_test(
    violations
):

    violations = np.asarray(
        violations
    )

    if len(violations) < 2:

        return {
            "LR_ind": np.nan,
            "p_value": np.nan
        }

    n00, n01, n10, n11 = (
        transition_counts(
            violations
        )
    )

    eps = 1e-12

    denominator0 = (
        n00 + n01
    )

    denominator1 = (
        n10 + n11
    )

    total_transitions = (
        n00
        +
        n01
        +
        n10
        +
        n11
    )

    total_violations = (
        n01
        +
        n11
    )

    if (
        denominator0 == 0
        or
        denominator1 == 0
        or
        total_transitions == 0
    ):

        return {
            "LR_ind": np.nan,
            "p_value": np.nan
        }

    pi01 = (
        n01
        /
        denominator0
    )

    pi11 = (
        n11
        /
        denominator1
    )

    pi = (
        total_violations
        /
        total_transitions
    )

    pi01 = np.clip(
        pi01,
        eps,
        1 - eps
    )

    pi11 = np.clip(
        pi11,
        eps,
        1 - eps
    )

    pi = np.clip(
        pi,
        eps,
        1 - eps
    )

    log_l_independent = (
        (n00 + n10)
        *
        np.log(1 - pi)
        +
        (n01 + n11)
        *
        np.log(pi)
    )

    log_l_markov = (
        n00
        *
        np.log(1 - pi01)
        +
        n01
        *
        np.log(pi01)
        +
        n10
        *
        np.log(1 - pi11)
        +
        n11
        *
        np.log(pi11)
    )

    LR_ind = (
        -2
        *
        (
            log_l_independent
            -
            log_l_markov
        )
    )

    p_value = (
        1
        -
        stats.chi2.cdf(
            LR_ind,
            df=1
        )
    )

    return {
        "LR_ind": LR_ind,
        "p_value": p_value
    }

In [12]:
christoffersen_example = (
    christoffersen_independence_test(
        violations
    )
)

christoffersen_example

{'LR_ind': np.float64(0.8396031934180144),
 'p_value': np.float64(0.35951028235403315)}

In [13]:
def conditional_coverage_test(
    violations,
    alpha
):

    kupiec = kupiec_test(
        violations,
        alpha
    )

    independence = (
        christoffersen_independence_test(
            violations
        )
    )

    if (
        np.isnan(
            kupiec["LR_uc"]
        )
        or
        np.isnan(
            independence["LR_ind"]
        )
    ):

        return {
            "LR_cc": np.nan,
            "p_value": np.nan
        }

    LR_cc = (
        kupiec["LR_uc"]
        +
        independence["LR_ind"]
    )

    p_value = (
        1
        -
        stats.chi2.cdf(
            LR_cc,
            df=2
        )
    )

    return {
        "LR_cc": LR_cc,
        "p_value": p_value
    }

In [14]:
cc_example = (
    conditional_coverage_test(
        violations,
        alpha=0.05
    )
)

cc_example

{'LR_cc': np.float64(19.66934320554509),
 'p_value': np.float64(5.356195146033471e-05)}

In [15]:
backtest_95_results = []

for _, row in oos_results.iterrows():

    dataset_name = row["Dataset"]

    model_name = row["Model"]

    actual_returns = load_test_returns(
        dataset_name
    )

    violations = violation_sequence(
        actual_returns,
        row["VaR_95"]
    )

    kupiec = kupiec_test(
        violations,
        alpha=0.05
    )

    christoffersen = (
        christoffersen_independence_test(
            violations
        )
    )

    conditional = (
        conditional_coverage_test(
            violations,
            alpha=0.05
        )
    )

    backtest_95_results.append({

        "Dataset": dataset_name,

        "Model": model_name,

        "Confidence": "95%",

        "Observations":
            len(violations),

        "Violations":
            int(
                violations.sum()
            ),

        "Violation_Rate":
            violations.mean(),

        "Expected_Rate":
            0.05,

        "Kupiec_LR":
            kupiec["LR_uc"],

        "Kupiec_p":
            kupiec["p_value"],

        "Christoffersen_LR":
            christoffersen[
                "LR_ind"
            ],

        "Christoffersen_p":
            christoffersen[
                "p_value"
            ],

        "Conditional_LR":
            conditional[
                "LR_cc"
            ],

        "Conditional_p":
            conditional[
                "p_value"
            ]
    })

In [16]:
backtest_95 = pd.DataFrame(
    backtest_95_results
)

backtest_95

,Dataset,Model,Confidence,Observations,Violations,Violation_Rate,Expected_Rate,Kupiec_LR,Kupiec_p,Christoffersen_LR,Christoffersen_p,Conditional_LR,Conditional_p
0,Short_Normal,Single-Dependence Gaussian,95%,880,19,0.021591,0.05,18.829740,1.429211e-05,0.839603,3.595103e-01,19.669343,5.356195e-05
1,Short_Normal,Regime-Horizon Student-t,95%,880,21,0.023864,0.05,15.561028,7.988429e-05,1.028075,3.106113e-01,16.589103,2.498745e-04
2,Short_Stress,Single-Dependence Gaussian,95%,59,1,0.016949,0.05,1.803458,1.792951e-01,0.035090,8.514082e-01,1.838547,3.988087e-01
3,Short_Stress,Regime-Horizon Student-t,95%,59,1,0.016949,0.05,1.803458,1.792951e-01,0.035090,8.514082e-01,1.838547,3.988087e-01
4,Medium_Normal,Single-Dependence Gaussian,95%,880,18,0.020455,0.05,20.622914,5.592276e-06,41.887522,9.667767e-11,62.510436,2.664535e-14
5,Medium_Normal,Regime-Horizon Student-t,95%,880,22,0.025000,0.05,14.075459,1.756203e-04,41.235749,1.349321e-10,55.311208,9.756640e-13
6,Medium_Stress,Single-Dependence Gaussian,95%,59,2,0.033898,0.05,0.361380,5.477411e-01,4.594083,3.208249e-02,4.955463,8.393341e-02
7,Medium_Stress,Regime-Horizon Student-t,95%,59,2,0.033898,0.05,0.361380,5.477411e-01,4.594083,3.208249e-02,4.955463,8.393341e-02
8,Long_Normal,Single-Dependence Gaussian,95%,880,12,0.013636,0.05,34.026753,5.435950e-09,104.475517,0.000000e+00,138.502270,0.000000e+00
9,Long_Normal,Regime-Horizon Student-t,95%,880,12,0.013636,0.05,34.026753,5.435950e-09,104.475517,0.000000e+00,138.502270,0.000000e+00


In [17]:
backtest_99_results = []

for _, row in oos_results.iterrows():

    dataset_name = row["Dataset"]

    model_name = row["Model"]

    actual_returns = load_test_returns(
        dataset_name
    )

    violations = violation_sequence(
        actual_returns,
        row["VaR_99"]
    )

    kupiec = kupiec_test(
        violations,
        alpha=0.01
    )

    christoffersen = (
        christoffersen_independence_test(
            violations
        )
    )

    conditional = (
        conditional_coverage_test(
            violations,
            alpha=0.01
        )
    )

    backtest_99_results.append({

        "Dataset": dataset_name,

        "Model": model_name,

        "Confidence": "99%",

        "Observations":
            len(violations),

        "Violations":
            int(
                violations.sum()
            ),

        "Violation_Rate":
            violations.mean(),

        "Expected_Rate":
            0.01,

        "Kupiec_LR":
            kupiec["LR_uc"],

        "Kupiec_p":
            kupiec["p_value"],

        "Christoffersen_LR":
            christoffersen[
                "LR_ind"
            ],

        "Christoffersen_p":
            christoffersen[
                "p_value"
            ],

        "Conditional_LR":
            conditional[
                "LR_cc"
            ],

        "Conditional_p":
            conditional[
                "p_value"
            ]
    })

In [18]:
backtest_99 = pd.DataFrame(
    backtest_99_results
)

backtest_99

,Dataset,Model,Confidence,Observations,Violations,Violation_Rate,Expected_Rate,Kupiec_LR,Kupiec_p,Christoffersen_LR,Christoffersen_p,Conditional_LR,Conditional_p
0,Short_Normal,Single-Dependence Gaussian,99%,880,1,0.001136,0.01,11.320124,0.000767,0.002278,9.619335e-01,11.322402,3.478337e-03
1,Short_Normal,Regime-Horizon Student-t,99%,880,1,0.001136,0.01,11.320124,0.000767,0.002278,9.619335e-01,11.322402,3.478337e-03
2,Short_Stress,Single-Dependence Gaussian,99%,59,0,0.000000,0.01,1.185940,0.276150,NaN,NaN,NaN,NaN
3,Short_Stress,Regime-Horizon Student-t,99%,59,0,0.000000,0.01,1.185940,0.276150,NaN,NaN,NaN,NaN
4,Medium_Normal,Single-Dependence Gaussian,99%,880,2,0.002273,0.01,7.726520,0.005442,10.013535,1.553940e-03,17.740055,1.405387e-04
5,Medium_Normal,Regime-Horizon Student-t,99%,880,2,0.002273,0.01,7.726520,0.005442,10.013535,1.553940e-03,17.740055,1.405387e-04
6,Medium_Stress,Single-Dependence Gaussian,99%,59,0,0.000000,0.01,1.185940,0.276150,NaN,NaN,NaN,NaN
7,Medium_Stress,Regime-Horizon Student-t,99%,59,0,0.000000,0.01,1.185940,0.276150,NaN,NaN,NaN,NaN
8,Long_Normal,Single-Dependence Gaussian,99%,880,4,0.004545,0.01,3.318739,0.068494,31.075708,2.481576e-08,34.394447,3.398918e-08
9,Long_Normal,Regime-Horizon Student-t,99%,880,3,0.003409,0.01,5.181691,0.022826,20.702110,5.365685e-06,25.883801,2.395543e-06


In [19]:
backtest_all = pd.concat(
    [
        backtest_95,
        backtest_99
    ],
    ignore_index=True
)

backtest_all.head()

,Dataset,Model,Confidence,Observations,Violations,Violation_Rate,Expected_Rate,Kupiec_LR,Kupiec_p,Christoffersen_LR,Christoffersen_p,Conditional_LR,Conditional_p
0,Short_Normal,Single-Dependence Gaussian,95%,880,19,0.021591,0.05,18.829740,0.000014,0.839603,3.595103e-01,19.669343,5.356195e-05
1,Short_Normal,Regime-Horizon Student-t,95%,880,21,0.023864,0.05,15.561028,0.000080,1.028075,3.106113e-01,16.589103,2.498745e-04
2,Short_Stress,Single-Dependence Gaussian,95%,59,1,0.016949,0.05,1.803458,0.179295,0.035090,8.514082e-01,1.838547,3.988087e-01
3,Short_Stress,Regime-Horizon Student-t,95%,59,1,0.016949,0.05,1.803458,0.179295,0.035090,8.514082e-01,1.838547,3.988087e-01
4,Medium_Normal,Single-Dependence Gaussian,95%,880,18,0.020455,0.05,20.622914,0.000006,41.887522,9.667767e-11,62.510436,2.664535e-14


In [20]:
backtest_all[
    "Kupiec_Pass"
] = (
    backtest_all[
        "Kupiec_p"
    ]
    >=
    0.05
)

backtest_all[
    "Independence_Pass"
] = (
    backtest_all[
        "Christoffersen_p"
    ]
    >=
    0.05
)

backtest_all[
    "Conditional_Pass"
] = (
    backtest_all[
        "Conditional_p"
    ]
    >=
    0.05
)

In [21]:
backtest_all[
    [
        "Dataset",
        "Model",
        "Confidence",
        "Violation_Rate",
        "Kupiec_p",
        "Christoffersen_p",
        "Conditional_p",
        "Kupiec_Pass",
        "Independence_Pass",
        "Conditional_Pass"
    ]
]

,Dataset,Model,Confidence,Violation_Rate,Kupiec_p,Christoffersen_p,Conditional_p,Kupiec_Pass,Independence_Pass,Conditional_Pass
0,Short_Normal,Single-Dependence Gaussian,95%,0.021591,1.429211e-05,3.595103e-01,5.356195e-05,False,True,False
1,Short_Normal,Regime-Horizon Student-t,95%,0.023864,7.988429e-05,3.106113e-01,2.498745e-04,False,True,False
2,Short_Stress,Single-Dependence Gaussian,95%,0.016949,1.792951e-01,8.514082e-01,3.988087e-01,True,True,True
3,Short_Stress,Regime-Horizon Student-t,95%,0.016949,1.792951e-01,8.514082e-01,3.988087e-01,True,True,True
4,Medium_Normal,Single-Dependence Gaussian,95%,0.020455,5.592276e-06,9.667767e-11,2.664535e-14,False,False,False
5,Medium_Normal,Regime-Horizon Student-t,95%,0.025000,1.756203e-04,1.349321e-10,9.756640e-13,False,False,False
6,Medium_Stress,Single-Dependence Gaussian,95%,0.033898,5.477411e-01,3.208249e-02,8.393341e-02,True,False,True
7,Medium_Stress,Regime-Horizon Student-t,95%,0.033898,5.477411e-01,3.208249e-02,8.393341e-02,True,False,True
8,Long_Normal,Single-Dependence Gaussian,95%,0.013636,5.435950e-09,0.000000e+00,0.000000e+00,False,False,False
9,Long_Normal,Regime-Horizon Student-t,95%,0.013636,5.435950e-09,0.000000e+00,0.000000e+00,False,False,False


In [22]:
pass_summary = (
    backtest_all
    .groupby(
        [
            "Model",
            "Confidence"
        ]
    )[
        [
            "Kupiec_Pass",
            "Independence_Pass",
            "Conditional_Pass"
        ]
    ]
    .sum()
)

pass_summary

Kupiec_Pass  Independence_Pass  \
Model                      Confidence                                   
Regime-Horizon Student-t   95%                   2                  2   
                           99%                   3                  1   
Single-Dependence Gaussian 95%                   2                  2   
                           99%                   4                  1   

                                       Conditional_Pass  
Model                      Confidence                    
Regime-Horizon Student-t   95%                        2  
                           99%                        0  
Single-Dependence Gaussian 95%                        2  
                           99%                        0

In [23]:
backtest_all[
    "Absolute_Violation_Error"
] = (
    backtest_all[
        "Violation_Rate"
    ]
    -
    backtest_all[
        "Expected_Rate"
    ]
).abs()

In [24]:
calibration_summary = (
    backtest_all
    .groupby(
        [
            "Model",
            "Confidence"
        ]
    )[
        "Absolute_Violation_Error"
    ]
    .mean()
)

calibration_summary

Model                       Confidence
Regime-Horizon Student-t    95%           0.031109
                            99%           0.008864
Single-Dependence Gaussian  95%           0.032245
                            99%           0.008674
Name: Absolute_Violation_Error, dtype: float64

In [25]:
backtest_all[
    "Regime"
] = (
    backtest_all[
        "Dataset"
    ]
    .str.split("_")
    .str[1]
)

In [26]:
backtest_all[
    "Horizon"
] = (
    backtest_all[
        "Dataset"
    ]
    .str.split("_")
    .str[0]
)

In [27]:
backtest_all.head()

,Dataset,Model,Confidence,Observations,Violations,Violation_Rate,Expected_Rate,Kupiec_LR,Kupiec_p,Christoffersen_LR,Christoffersen_p,Conditional_LR,Conditional_p,Kupiec_Pass,Independence_Pass,Conditional_Pass,Absolute_Violation_Error,Regime,Horizon
0,Short_Normal,Single-Dependence Gaussian,95%,880,19,0.021591,0.05,18.829740,0.000014,0.839603,3.595103e-01,19.669343,5.356195e-05,False,True,False,0.028409,Normal,Short
1,Short_Normal,Regime-Horizon Student-t,95%,880,21,0.023864,0.05,15.561028,0.000080,1.028075,3.106113e-01,16.589103,2.498745e-04,False,True,False,0.026136,Normal,Short
2,Short_Stress,Single-Dependence Gaussian,95%,59,1,0.016949,0.05,1.803458,0.179295,0.035090,8.514082e-01,1.838547,3.988087e-01,True,True,True,0.033051,Stress,Short
3,Short_Stress,Regime-Horizon Student-t,95%,59,1,0.016949,0.05,1.803458,0.179295,0.035090,8.514082e-01,1.838547,3.988087e-01,True,True,True,0.033051,Stress,Short
4,Medium_Normal,Single-Dependence Gaussian,95%,880,18,0.020455,0.05,20.622914,0.000006,41.887522,9.667767e-11,62.510436,2.664535e-14,False,False,False,0.029545,Normal,Medium


In [28]:
stress_backtests = (
    backtest_all[
        backtest_all[
            "Regime"
        ]
        ==
        "Stress"
    ]
)

stress_summary = (
    stress_backtests
    .groupby(
        [
            "Model",
            "Confidence"
        ]
    )
    .agg(
        Mean_Violation_Rate=(
            "Violation_Rate",
            "mean"
        ),
        Mean_Absolute_Error=(
            "Absolute_Violation_Error",
            "mean"
        ),
        Kupiec_Passes=(
            "Kupiec_Pass",
            "sum"
        ),
        Conditional_Passes=(
            "Conditional_Pass",
            "sum"
        )
    )
)

stress_summary

Mean_Violation_Rate  \
Model                      Confidence                        
Regime-Horizon Student-t   95%                    0.016949   
                           99%                    0.000000   
Single-Dependence Gaussian 95%                    0.016949   
                           99%                    0.000000   

                                       Mean_Absolute_Error  Kupiec_Passes  \
Model                      Confidence                                       
Regime-Horizon Student-t   95%                    0.033051              2   
                           99%                    0.010000              3   
Single-Dependence Gaussian 95%                    0.033051              2   
                           99%                    0.010000              3   

                                       Conditional_Passes  
Model                      Confidence                      
Regime-Horizon Student-t   95%                          2  
                           99%                          0  
Single-Dependence Gaussian 95%                          2  
                           99%                          0

In [29]:
normal_backtests = (
    backtest_all[
        backtest_all[
            "Regime"
        ]
        ==
        "Normal"
    ]
)

normal_summary = (
    normal_backtests
    .groupby(
        [
            "Model",
            "Confidence"
        ]
    )
    .agg(
        Mean_Violation_Rate=(
            "Violation_Rate",
            "mean"
        ),
        Mean_Absolute_Error=(
            "Absolute_Violation_Error",
            "mean"
        ),
        Kupiec_Passes=(
            "Kupiec_Pass",
            "sum"
        ),
        Conditional_Passes=(
            "Conditional_Pass",
            "sum"
        )
    )
)

normal_summary

Mean_Violation_Rate  \
Model                      Confidence                        
Regime-Horizon Student-t   95%                    0.020833   
                           99%                    0.002273   
Single-Dependence Gaussian 95%                    0.018561   
                           99%                    0.002652   

                                       Mean_Absolute_Error  Kupiec_Passes  \
Model                      Confidence                                       
Regime-Horizon Student-t   95%                    0.029167              0   
                           99%                    0.007727              0   
Single-Dependence Gaussian 95%                    0.031439              0   
                           99%                    0.007348              1   

                                       Conditional_Passes  
Model                      Confidence                      
Regime-Horizon Student-t   95%                          0  
                           99%                          0  
Single-Dependence Gaussian 95%                          0  
                           99%                          0

In [30]:
final_backtest_table = (
    backtest_all[
        [
            "Dataset",
            "Model",
            "Confidence",
            "Observations",
            "Violations",
            "Violation_Rate",
            "Expected_Rate",
            "Kupiec_p",
            "Christoffersen_p",
            "Conditional_p",
            "Kupiec_Pass",
            "Independence_Pass",
            "Conditional_Pass"
        ]
    ]
)

final_backtest_table

,Dataset,Model,Confidence,Observations,Violations,Violation_Rate,Expected_Rate,Kupiec_p,Christoffersen_p,Conditional_p,Kupiec_Pass,Independence_Pass,Conditional_Pass
0,Short_Normal,Single-Dependence Gaussian,95%,880,19,0.021591,0.05,1.429211e-05,3.595103e-01,5.356195e-05,False,True,False
1,Short_Normal,Regime-Horizon Student-t,95%,880,21,0.023864,0.05,7.988429e-05,3.106113e-01,2.498745e-04,False,True,False
2,Short_Stress,Single-Dependence Gaussian,95%,59,1,0.016949,0.05,1.792951e-01,8.514082e-01,3.988087e-01,True,True,True
3,Short_Stress,Regime-Horizon Student-t,95%,59,1,0.016949,0.05,1.792951e-01,8.514082e-01,3.988087e-01,True,True,True
4,Medium_Normal,Single-Dependence Gaussian,95%,880,18,0.020455,0.05,5.592276e-06,9.667767e-11,2.664535e-14,False,False,False
5,Medium_Normal,Regime-Horizon Student-t,95%,880,22,0.025000,0.05,1.756203e-04,1.349321e-10,9.756640e-13,False,False,False
6,Medium_Stress,Single-Dependence Gaussian,95%,59,2,0.033898,0.05,5.477411e-01,3.208249e-02,8.393341e-02,True,False,True
7,Medium_Stress,Regime-Horizon Student-t,95%,59,2,0.033898,0.05,5.477411e-01,3.208249e-02,8.393341e-02,True,False,True
8,Long_Normal,Single-Dependence Gaussian,95%,880,12,0.013636,0.05,5.435950e-09,0.000000e+00,0.000000e+00,False,False,False
9,Long_Normal,Regime-Horizon Student-t,95%,880,12,0.013636,0.05,5.435950e-09,0.000000e+00,0.000000e+00,False,False,False


In [31]:
display_backtest = (
    final_backtest_table.copy()
)

numeric_cols = [
    "Violation_Rate",
    "Expected_Rate",
    "Kupiec_p",
    "Christoffersen_p",
    "Conditional_p"
]

display_backtest[
    numeric_cols
] = (
    display_backtest[
        numeric_cols
    ]
    .round(4)
)

display_backtest

,Dataset,Model,Confidence,Observations,Violations,Violation_Rate,Expected_Rate,Kupiec_p,Christoffersen_p,Conditional_p,Kupiec_Pass,Independence_Pass,Conditional_Pass
0,Short_Normal,Single-Dependence Gaussian,95%,880,19,0.0216,0.05,0.0000,0.3595,0.0001,False,True,False
1,Short_Normal,Regime-Horizon Student-t,95%,880,21,0.0239,0.05,0.0001,0.3106,0.0002,False,True,False
2,Short_Stress,Single-Dependence Gaussian,95%,59,1,0.0169,0.05,0.1793,0.8514,0.3988,True,True,True
3,Short_Stress,Regime-Horizon Student-t,95%,59,1,0.0169,0.05,0.1793,0.8514,0.3988,True,True,True
4,Medium_Normal,Single-Dependence Gaussian,95%,880,18,0.0205,0.05,0.0000,0.0000,0.0000,False,False,False
5,Medium_Normal,Regime-Horizon Student-t,95%,880,22,0.0250,0.05,0.0002,0.0000,0.0000,False,False,False
6,Medium_Stress,Single-Dependence Gaussian,95%,59,2,0.0339,0.05,0.5477,0.0321,0.0839,True,False,True
7,Medium_Stress,Regime-Horizon Student-t,95%,59,2,0.0339,0.05,0.5477,0.0321,0.0839,True,False,True
8,Long_Normal,Single-Dependence Gaussian,95%,880,12,0.0136,0.05,0.0000,0.0000,0.0000,False,False,False
9,Long_Normal,Regime-Horizon Student-t,95%,880,12,0.0136,0.05,0.0000,0.0000,0.0000,False,False,False


In [32]:
backtest_95.to_csv(
    "outputs/tables/backtest_95.csv",
    index=False
)

backtest_99.to_csv(
    "outputs/tables/backtest_99.csv",
    index=False
)

backtest_all.to_csv(
    "outputs/tables/backtest_all.csv",
    index=False
)

final_backtest_table.to_csv(
    "outputs/tables/final_backtest_table.csv",
    index=False
)

stress_summary.to_csv(
    "outputs/tables/stress_backtest_summary.csv"
)

normal_summary.to_csv(
    "outputs/tables/normal_backtest_summary.csv"
)

pass_summary.to_csv(
    "outputs/tables/backtest_pass_summary.csv"
)

In [33]:
print(
    "Notebook 10 completed successfully."
)

print(
    "\nPass summary:"
)

display(
    pass_summary
)

print(
    "\nStress-period summary:"
)

display(
    stress_summary.round(4)
)

print(
    "\nNormal-period summary:"
)

display(
    normal_summary.round(4)
)

print(
    "\nCalibration summary:"
)

display(
    calibration_summary.round(4)
)

Notebook 10 completed successfully.

Pass summary:


Kupiec_Pass  Independence_Pass  \
Model                      Confidence                                   
Regime-Horizon Student-t   95%                   2                  2   
                           99%                   3                  1   
Single-Dependence Gaussian 95%                   2                  2   
                           99%                   4                  1   

                                       Conditional_Pass  
Model                      Confidence                    
Regime-Horizon Student-t   95%                        2  
                           99%                        0  
Single-Dependence Gaussian 95%                        2  
                           99%                        0


Stress-period summary:


Mean_Violation_Rate  \
Model                      Confidence                        
Regime-Horizon Student-t   95%                      0.0169   
                           99%                      0.0000   
Single-Dependence Gaussian 95%                      0.0169   
                           99%                      0.0000   

                                       Mean_Absolute_Error  Kupiec_Passes  \
Model                      Confidence                                       
Regime-Horizon Student-t   95%                      0.0331              2   
                           99%                      0.0100              3   
Single-Dependence Gaussian 95%                      0.0331              2   
                           99%                      0.0100              3   

                                       Conditional_Passes  
Model                      Confidence                      
Regime-Horizon Student-t   95%                          2  
                           99%                          0  
Single-Dependence Gaussian 95%                          2  
                           99%                          0


Normal-period summary:


Mean_Violation_Rate  \
Model                      Confidence                        
Regime-Horizon Student-t   95%                      0.0208   
                           99%                      0.0023   
Single-Dependence Gaussian 95%                      0.0186   
                           99%                      0.0027   

                                       Mean_Absolute_Error  Kupiec_Passes  \
Model                      Confidence                                       
Regime-Horizon Student-t   95%                      0.0292              0   
                           99%                      0.0077              0   
Single-Dependence Gaussian 95%                      0.0314              0   
                           99%                      0.0073              1   

                                       Conditional_Passes  
Model                      Confidence                      
Regime-Horizon Student-t   95%                          0  
                           99%                          0  
Single-Dependence Gaussian 95%                          0  
                           99%                          0


Calibration summary:


Model                       Confidence
Regime-Horizon Student-t    95%           0.0311
                            99%           0.0089
Single-Dependence Gaussian  95%           0.0322
                            99%           0.0087
Name: Absolute_Violation_Error, dtype: float64